In [5]:
%pip install spacy
%pip install pandas

  Using cached spacy-3.8.2.tar.gz (1.3 MB)
  Installing build dependencies ... error
  ERROR: Command errored out with exit status 1:
   command: /home/pianotriescode/.pyenv/versions/3.8.7/bin/python /home/pianotriescode/.pyenv/versions/3.8.7/lib/python3.8/site-packages/pip install --ignore-installed --no-user --prefix /tmp/pip-build-env-lclfxepm/overlay --no-warn-script-location --no-binary :none: --only-binary :none: -i https://pypi.org/simple -- setuptools 'cython>=0.25,<3.0' 'cymem>=2.0.2,<2.1.0' 'preshed>=3.0.2,<3.1.0' 'murmurhash>=0.28.0,<1.1.0' 'thinc>=8.3.0,<8.4.0' 'numpy>=2.0.0,<2.1.0; python_version < '"'"'3.9'"'"'' 'numpy>=2.0.0,<2.1.0; python_version >= '"'"'3.9'"'"''
       cwd: None
  Complete output (71 lines):
  Ignoring numpy: markers 'python_version >= "3.9"' don't match your environment
    Installing build dependencies: started
    Installing build dependencies: finished with status 'done'
    Getting requirements to build wheel: started
    Getting requirements to 

In [2]:
import spacy
import pandas as pd
nlp = spacy.load("en_ner_bc5cdr_md")


/home/pianotriescode/Hallucination/Reducing-Hallucinations-in-Clinical-Diagnosis/venv/lib/python3.10/site-packages/torch/__init__.py:1240: UserWarning: torch.set_default_tensor_type() is deprecated as of PyTorch 2.1, please use torch.set_default_dtype() and torch.set_default_device() as alternatives. (Triggered internally at /pytorch/torch/csrc/tensor/python_tensor.cpp:434.)
  _C._set_default_tensor_type(t)


In [3]:
df = pd.read_csv("../data/challenge_data/clinicalnlp_taskB_test1.csv")
df.head()

,dataset,encounter_id,dialogue,note
0,virtassist,D2N088,"[doctor] hi , andrew . how are you ?\n[patient...",CHIEF COMPLAINT\n\nUpper respiratory infection...
1,virtassist,D2N089,"[doctor] hi andrea , how are you ?\n[patient] ...",CHIEF COMPLAINT\n\nAnnual exam.\n\nHISTORY OF ...
2,virtassist,D2N090,"[doctor] hi , albert . how are you ?\n[patient...",CHIEF COMPLAINT\n\nER follow-up.\n\nHISTORY OF...
3,virtassist,D2N091,"[doctor] hi jerry , how are you doing ?\n[pati...",CHIEF COMPLAINT\n\nAnnual exam.\n\nHISTORY OF ...
4,virtassist,D2N092,"[doctor] hello , mrs . martinez . good to see ...",CC:\n\nRight arm pain.\n\nHPI:\n\nMs. Martinez...


In [5]:
len(df)

40

In [4]:
for note in df['note']:
    doc = nlp(note)
    for ent in doc.ents:
        print(ent.text, ent.label_)

respiratory infection DISEASE
depression DISEASE
diabetes DISEASE
hypertension DISEASE
upper respiratory infection DISEASE
phlegm DISEASE
fever DISEASE
allergies DISEASE
depression DISEASE
diabetes DISEASE
lisinopril CHEMICAL
nausea DISEASE
vomiting DISEASE
diarrhea DISEASE
Denies fever DISEASE
dyspnea DISEASE
shortness of breath DISEASE
cough DISEASE
Denies nausea or diarrhea DISEASE
pain DISEASE
Psychiatric DISEASE
Endorses depression DISEASE
rhonchi DISEASE
cough DISEASE
murmurs DISEASE
Edema DISEASE
lower extremities DISEASE
Pain DISEASE
chest DISEASE
airspace disease DISEASE
pneumonia DISEASE
depression DISEASE
diabetes DISEASE
hypertension DISEASE
upper respiratory infection DISEASE
respiratory infection DISEASE
viral syndrome DISEASE
Robitussin CHEMICAL
cough DISEASE
ibuprofen CHEMICAL
Tylenol CHEMICAL
fever DISEASE
Depression DISEASE
Diabetes DISEASE
glucose CHEMICAL
metformin CHEMICAL
Hypertension DISEASE
lisinopril CHEMICAL
lisinopril CHEMICAL
rheumatoid arthritis DISEASE
atr

In [ ]:
from utils import get_model
import os
import json
os.environ["CUDA_VISIBLE_DEVICES"] = "7"
model_type = "7b"
model_family = "llamabase"

wiki_path = "./auto-labeled/wiki"
output_path = f"./auto-labeled/output/{model_family}{model_type}"

topk_first_token = 4
windows = 16
if "llama" in model_family or "baichuan" in model_family:
    st = "▁"
else:
    st = "Ġ"

# Define medical entity types to focus on
MEDICAL_ENTITY_TYPES = {
    'DISEASE', 'SYMPTOM', 'ANATOMY', 'CHEMICAL', 'PHARMACEUTICAL', 
    'PROCEDURE', 'DEVICE', 'CONDITION'
}

prompt_chat = []

def delete_substrings(lst):
    substrings = []
    lst = list(set(lst))
    for s in lst:
        if any(s in o for o in lst if o != s):
            substrings.append(s)
    for s in substrings:
        lst.remove(s)
    return lst

def find_boundaries(text, words):
    boundaries = []
    for word in words:
        start = 0
        ntext = text
        while True:
            start = ntext.find(word)
            if start == -1:
                break
            end = start + len(word) - 1
            while start > 0 and ntext[start-1] != " ":
                start -= 1
            while end < len(ntext)-1 and ntext[end+1] != " ":
                end += 1
            boundaries.append("".join([ntext[i] for i in range(start, end+1)]))
            ntext = ntext[end+1:]
    return boundaries


def get_entities(note):
    doc = nlp(note)
    entities = [ent.text for ent in doc.ents if ent.label_ in MEDICAL_ENTITY_TYPES]
    entities = list(set(entities))
    entities = find_boundaries(note,entities)
    entities = delete_substrings(entities)

    all_entities = []
    for i in range(len(note)):
        for e in entities:
            if note[i:].startswith(e):
                all_entities.append((e, i))
                
    return all_entities

with open("/data/challenge_data/clinicalnlp_taskB_test1.csv", encoding='utf-8') as f:
        data = json.load(f)

for ii, d in enumerate(data):

    text = " ".join(d["sentences"][:2])
    entities_ = []
    entities_ += get_entities(text)
    
    entities = []
    idx_ = []
    for e in entities_:
        if e[1] not in idx_:
            idx_.append(e[1])
            entities.append(e)

    mytexts = []
    new_entities = []
    original_entity = []
    ret = {
            "original_text": text,
            "title": d["title"]
        }


SyntaxError: incomplete input (4222789199.py, line 55)